# 🔬 vLLM 源码级深度剖析

> **核心命题**：如何让 GPU 显存在高并发推理时不碎片、不浪费？

vLLM 的论文标题非常直接：*"Efficient Memory Management for Large Language Model Serving with PagedAttention"*。
它解决的问题不是"如何算得更快"，而是"**如何让显存放得下更多请求**"。

## 问题：传统 KV Cache 管理的灾难

在 vLLM 之前，推理框架（包括早期的 HuggingFace TGI）是怎么管理 KV Cache 的？

```
请求 A (max_len=512):   [████████████████████████████████████████]  预分配 512 × KV_size
请求 B (max_len=2048):  [████████████████████...留空████████████████]  预分配 2048 × KV_size
请求 C (max_len=128):   [██...留空██████████████████████████████████]  预分配 128 × KV_size

问题：
  1. 内部碎片：请求 C 实际生成了 50 token，但分配了 128 的位置 → 61% 浪费
  2. 外部碎片：请求完成后释放的空间不连续，无法分配给更大的请求
  3. 无法共享：3 个请求有相同的 system prompt，KV Cache 各自存一份 → 3× 浪费
```

vLLM 的洞察：**这和操作系统管理物理内存的问题一模一样**。
所以答案也一模一样：**分页 + 虚拟地址映射**。

## PagedAttention 的核心思想

```
传统:                         PagedAttention:
  KV Cache 连续存储              KV Cache 分块存储 (block = 16 tokens)
                               
  Req A: ████████░░░░           Req A: [B0]→[B1]→[B5]→[B3]
  Req B: ████████████████░           Block Table: [0, 1, 5, 3]
  Req C: ████░░░░░░░░░░░                   ↓
                                     使用 block_table 间接寻址
  ↑ 内部碎片                        
  不能共享                          Req A & B 共享 system prompt:
                                     [B0: "You are a helpful"] ← 共享
                                     [B1: "assistant. Answer"]  ← 共享
                                     Req A: [B0, B1, B5, B3]
                                     Req B: [B0, B1, B7, B2]
                                     ↑ B0, B1 引用计数 = 2，不会被释放
```

## 1. 架构全景

```
┌──────────────────────────────────────────────────────────────────┐
│                         vLLM 架构                                │
├──────────────────────────────────────────────────────────────────┤
│                                                                   │
│  ┌─────────────────────────────────────────────────────────────┐ │
│  │                    API Server (FastAPI)                      │ │
│  │  /v1/chat/completions  /v1/completions  (OpenAI 兼容)       │ │
│  └─────────────────────────────┬───────────────────────────────┘ │
│                                │                                  │
│  ┌─────────────────────────────▼───────────────────────────────┐ │
│  │                   Scheduler (调度器)                         │ │
│  │  • 请求队列管理 (WAITING → RUNNING → FINISHED)               │ │
│  │  • 决定每个 step 哪些请求参与计算 (continuous batching)       │
│  │  • 抢占策略 (preemption): 换出 KV blocks 到 CPU 内存         │
│  └─────────────────────────────┬───────────────────────────────┘ │
│                                │                                  │
│  ┌─────────────────────────────▼───────────────────────────────┐ │
│  │               BlockSpaceManager (块空间管理器)               │ │
│  │  • 物理块分配 / 释放 (类似 OS 的物理页帧分配)                │ │
│  │  • Block Table 维护 (逻辑块 → 物理块的映射)                  │ │
│  │  • 引用计数 / Copy-on-Write (前缀共享)                       │ │
│  └─────────────────────────────┬───────────────────────────────┘ │
│                                │                                  │
│  ┌─────────────────────────────▼───────────────────────────────┐ │
│  │              Model Runner (模型执行器)                        │ │
│  │  • 将多个请求的 token 打包成 batch                            │
│  │  • 调用 PagedAttention kernel (vLLM 自研 CUDA kernel)       │
│  │  • FlashAttention / FlashInfer 后端                          │
│  │  • 采样 (logits → next token)                                │
│  └─────────────────────────────────────────────────────────────┘ │
│                                                                   │
└──────────────────────────────────────────────────────────────────┘
```

### 关键源文件

| 文件 | 职责 |
|------|------|
| `vllm/core/scheduler.py` | 调度器核心：请求队列、batch 选择、抢占 |
| `vllm/core/block_manager.py` | 块空间管理：分配、释放、block_table |
| `vllm/core/block/` | 块管理器的多种实现 (naive / prefix_caching) |
| `vllm/worker/model_runner.py` | 模型执行：构建 batch、调用 kernel |
| `vllm/attention/` | PagedAttention kernel 的多种后端 |
| `vllm/model_executor/layers/attention.py` | 通用 Attention 层实现 |
| `csrc/cache_kernels.cu` | CUDA kernel：reshape_and_cache, copy_blocks |
| `csrc/attention/paged_attention.cu` | PagedAttention 的 CUDA kernel |

## 2. BlockSpaceManager：物理块与逻辑块的映射

这是 vLLM 最核心的数据结构，对应操作系统课程里"页表"的概念。

### 2.1 数据结构（简化自 vllm/core/block/）

```python
# 物理块 (Physical Block) = GPU 显存中的一段连续 KV Cache
# 每个物理块存储 block_size 个 token 的 K 和 V
# 对所有请求全局统一管理

class PhysicalBlock:
    block_id: int          # 全局唯一 ID，对应显存中的位置
    ref_count: int          # 引用计数 → 0 时可回收
    
class BlockTable:
    """逻辑块到物理块的映射表 — 类比 OS 的页表"""
    blocks: List[int]       # blocks[i] = physical_block_id
                             # 第 i 个逻辑块映射到哪个物理块

class BlockSpaceManager:
    """
    类比 OS 的物理内存管理器。
    全局有 num_gpu_blocks 个物理块，
    每个请求有自己的 BlockTable (页表)。
    """
    # === 全局状态 ===
    num_total_gpu_blocks: int       # GPU 上物理块总数
    free_blocks: Deque[int]         # 空闲块队列 (类似 OS 的空闲页帧)
    all_blocks: List[PhysicalBlock] # 所有物理块的状态
    
    # === 每个序列的状态 ===
    block_tables: Dict[SeqId, BlockTable]  # 序列 → 逻辑→物理映射
    
    # === 关键操作 ===
    
    def allocate(self, seq_id: SeqId, num_new_tokens: int) -> List[int]:
        """
        为一个序列分配新的物理块。
        类比 OS 的 mmap / sbrk。
        
        返回新分配的物理块 ID 列表
        """
        num_blocks_needed = (num_new_tokens + block_size - 1) // block_size
        allocated = []
        for _ in range(num_blocks_needed):
            if not self.free_blocks:
                raise OutOfMemoryError  # 触发抢占
            block_id = self.free_blocks.popleft()
            self.all_blocks[block_id].ref_count = 1
            self.block_tables[seq_id].append(block_id)
            allocated.append(block_id)
        return allocated
    
    def free(self, seq_id: SeqId):
        """
        释放一个序列占用的所有物理块。
        类比 OS 进程退出时回收页表。
        """
        for block_id in self.block_tables[seq_id]:
            self.all_blocks[block_id].ref_count -= 1
            if self.all_blocks[block_id].ref_count == 0:
                self.free_blocks.append(block_id)  # 回收
        del self.block_tables[seq_id]
    
    def fork(self, parent_seq_id: SeqId, child_seq_id: SeqId):
        """
        Copy-on-Write 分叉：子序列共享父序列的所有块。
        类比 OS 的 fork() + COW。
        
        关键场景：同一个 system prompt 被多个请求使用
        → 物理块只存一份，所有请求的 block_table 指向同一个物理块
        """
        child_table = self.block_tables[parent_seq_id].copy()
        for block_id in child_table:
            self.all_blocks[block_id].ref_count += 1  # 增加引用
        self.block_tables[child_seq_id] = child_table
```

In [ ]:
# 完整模拟 vLLM 的 BlockSpaceManager
from dataclasses import dataclass, field
from collections import deque
from typing import Dict, List, Optional

@dataclass
class PhysicalBlock:
    block_id: int
    ref_count: int = 0

@dataclass  
class Sequence:
    seq_id: int
    token_ids: List[int] = field(default_factory=list)
    block_table: List[int] = field(default_factory=list)  # 逻辑块→物理块
    status: str = "WAITING"  # WAITING → RUNNING → FINISHED

class BlockSpaceManager:
    """vLLM 的块空间管理器 — 核心数据结构"""
    
    def __init__(self, block_size: int, num_gpu_blocks: int):
        self.block_size = block_size
        self.num_gpu_blocks = num_gpu_blocks
        
        # 物理块池
        self.blocks = [PhysicalBlock(i) for i in range(num_gpu_blocks)]
        self.free_blocks = deque(range(num_gpu_blocks))
        
        # 序列管理
        self.sequences: Dict[int, Sequence] = {}
        
        # 统计
        self.total_allocations = 0
        self.total_cow_saved = 0
    
    def alloc(self, seq_id: int, num_tokens: int) -> bool:
        """为一个序列分配块，返回是否成功"""
        seq = self.sequences[seq_id]
        current_blocks = len(seq.block_table)
        total_blocks_needed = (num_tokens + self.block_size - 1) // self.block_size
        new_blocks_needed = total_blocks_needed - current_blocks
        
        if new_blocks_needed <= 0:
            return True
        
        if len(self.free_blocks) < new_blocks_needed:
            return False  # OOM — 调度器需要决定抢占
        
        for _ in range(new_blocks_needed):
            block_id = self.free_blocks.popleft()
            self.blocks[block_id].ref_count = 1
            seq.block_table.append(block_id)
        
        self.total_allocations += new_blocks_needed
        return True
    
    def fork(self, parent_id: int, child_id: int):
        """
        Copy-on-Write 分叉。
        新序列共享父序列的所有物理块。
        这是 prefix caching 的基础。
        """
        parent = self.sequences[parent_id]
        child = self.sequences[child_id]
        
        # 复制 block_table (逻辑映射)，增加物理块引用计数
        child.block_table = parent.block_table.copy()
        for block_id in child.block_table:
            self.blocks[block_id].ref_count += 1
        
        child.token_ids = parent.token_ids.copy()
    
    def _copy_on_write(self, seq_id: int, block_index: int) -> int:
        """
        COW 的实际写入触发。
        当共享块需要被修改时，分配新块并拷贝。
        """
        seq = self.sequences[seq_id]
        old_block = seq.block_table[block_index]
        
        if self.blocks[old_block].ref_count > 1:
            # 被多个序列共享 — 需要 COW
            if not self.free_blocks:
                return -1  # OOM
            
            new_block = self.free_blocks.popleft()
            self.blocks[new_block].ref_count = 1
            self.blocks[old_block].ref_count -= 1
            
            # 实际场景中这里要拷贝 GPU 上的 KV Cache 数据
            # copy_blocks_kernel(old_block, new_block)
            
            seq.block_table[block_index] = new_block
            self.total_cow_saved += 1  # 节省了一个块的分配（大部分块仍然是共享的）
            return new_block
        
        return old_block  # 只有一个引用，无需 COW
    
    def free_sequence(self, seq_id: int):
        """释放一个序列"""
        seq = self.sequences[seq_id]
        for block_id in seq.block_table:
            self.blocks[block_id].ref_count -= 1
            if self.blocks[block_id].ref_count == 0:
                self.free_blocks.append(block_id)
        seq.block_table = []
        seq.status = "FINISHED"
    
    def usage(self) -> Dict:
        used = sum(1 for b in self.blocks if b.ref_count > 0)
        shared = sum(1 for b in self.blocks if b.ref_count > 1)
        return {
            "total_blocks": self.num_gpu_blocks,
            "used_blocks": used,
            "free_blocks": len(self.free_blocks),
            "shared_blocks": shared,
            "usage_pct": f"{used/self.num_gpu_blocks*100:.1f}%"
        }

# === 演示 ===
mgr = BlockSpaceManager(block_size=16, num_gpu_blocks=100)

# 创建 3 个请求，共享同一个 system prompt (64 tokens)
system_prompt_tokens = list(range(64))  # 模拟 system prompt

mgr.sequences[1] = Sequence(seq_id=1)
mgr.sequences[2] = Sequence(seq_id=2)
mgr.sequences[3] = Sequence(seq_id=3)

# Seq 1 先创建并分配 system prompt
mgr.sequences[1].token_ids = system_prompt_tokens
mgr.alloc(1, len(system_prompt_tokens))  # 64 tokens → 4 blocks
print("After seq 1 (system prompt):")
print(f"  Block table: {mgr.sequences[1].block_table}")
print(f"  Usage: {mgr.usage()}")

# Seq 2 和 3 fork 自 Seq 1 — 共享 system prompt 的物理块！
mgr.fork(1, 2)  # COW! 不分配新块
mgr.fork(1, 3)  # COW! 不分配新块

print("\nAfter fork (seq 2 & 3 share seq 1's blocks):")
print(f"  Seq 1 block_table: {mgr.sequences[1].block_table}")
print(f"  Seq 2 block_table: {mgr.sequences[2].block_table}")
print(f"  Seq 3 block_table: {mgr.sequences[3].block_table}")
print(f"  Usage: {mgr.usage()}")

# 传统方式每个请求各自分配: 4 blocks × 3 = 12 blocks
# PagedAttention COW:   4 blocks (共享)  = 4 blocks
print(f"\n💡 节省: 传统 12 blocks vs PagedAttention 4 blocks → 节省 67% 显存")

# Seq 1 生成 32 个新 token — 需要分配新块，但不影响共享的 system prompt 块
mgr.sequences[1].token_ids.extend(range(32))
mgr.alloc(1, len(mgr.sequences[1].token_ids))  # 64+32=96 → 6 blocks

print(f"\nAfter seq 1 generates 32 more tokens:")
print(f"  Seq 1 block_table: {mgr.sequences[1].block_table}")
print(f"  前 4 块仍与 seq 2, seq 3 共享")
print(f"  Usage: {mgr.usage()}")


## 3. PagedAttention Kernel：如何在 GPU 上实现分页注意力

理解了数据结构的"分页"，还需要理解 GPU kernel 如何利用 block_table 做间接寻址。

### 3.1 传统 Attention vs PagedAttention (GPU 视角)

```cuda
// ===== 传统 Attention: 连续 KV Cache =====
// K_cache: [num_tokens, num_heads, head_dim]  — 连续存储
// 直接访问: K_cache[token_pos][head]
// 实现简单，但无法处理分块

// ===== PagedAttention: 分块 KV Cache =====
// 每个请求有一个 block_table[]，记录了它的逻辑块在哪里
// GPU kernel 需要:
//   1. 读 block_table[logical_block_id] → 得到 physical_block_id
//   2. 用 physical_block_id * block_size + offset → 找到实际的 K/V 地址
//   3. 计算 softmax(Q @ K^T / sqrt(d)) 时需要遍历所有逻辑块

__global__ void paged_attention_kernel(
    float* __restrict__ out,        // [num_seqs, num_heads, head_dim]
    const float* __restrict__ Q,    // [num_seqs, num_heads, head_dim]
    const float* __restrict__ K_cache,  // [num_blocks, num_heads, block_size, head_dim]
    const float* __restrict__ V_cache,  // [num_blocks, num_heads, block_size, head_dim]
    const int* __restrict__ block_tables, // [num_seqs, max_blocks_per_seq]
    const int* __restrict__ context_lens, // [num_seqs]
    int block_size,
    float scale
) {
    int seq_id = blockIdx.x;
    int head_id = blockIdx.y;
    int context_len = context_lens[seq_id];
    int num_blocks = (context_len + block_size - 1) / block_size;
    
    // 1. 加载该 head 的 Q 向量 [head_dim]
    float q_vec[HEAD_DIM];
    for (int d = threadIdx.x; d < head_dim; d += blockDim.x) {
        q_vec[d] = Q[seq_id * num_heads * head_dim + head_id * head_dim + d];
    }
    
    // 2. 遍历该序列的所有逻辑块，计算 attention
    float partial_sum = 0.0f;
    float max_logit = -INFINITY;
    float output[HEAD_DIM] = {0};
    
    for (int block_idx = 0; block_idx < num_blocks; block_idx++) {
        // 关键：间接寻址 —— 通过 block_table 找到物理块
        int physical_block = block_tables[seq_id * max_blocks_per_seq + block_idx];
        
        for (int t = 0; t < block_size; t++) {
            int global_t = block_idx * block_size + t;
            if (global_t >= context_len) break;
            
            // 读 K[physical_block, head, t, :]
            float qk = 0.0f;
            for (int d = 0; d < head_dim; d++) {
                float k_val = K_cache[physical_block * num_heads * block_size * head_dim
                                     + head_id * block_size * head_dim
                                     + t * head_dim + d];
                qk += q_vec[d] * k_val;
            }
            qk *= scale;
            
            // Online softmax (数值稳定的流式计算)
            float new_max = fmaxf(max_logit, qk);
            float correction = expf(max_logit - new_max);
            partial_sum = partial_sum * correction + expf(qk - new_max);
            max_logit = new_max;
        }
    }
    // ... 写入 output，使用 partial_sum 做归一化
}
```

**核心开销**：每读一次 K 或 V 都需要通过 `block_table` 做一次间接寻址。
但由于 block_size 通常为 16，同一个 block 内的 16 个 token 是连续存储的，
所以间接寻址的开销是 `num_blocks` 级别而非 `num_tokens` 级别。

In [ ]:
# Python 模拟 PagedAttention 的寻址逻辑
# 比 CUDA kernel 慢 1000×，但能直观展示 block_table 间接寻址

import numpy as np

class PagedAttentionSimulator:
    """模拟 GPU 上的 PagedAttention 寻址"""
    
    def __init__(self, num_blocks: int, num_heads: int, head_dim: int, block_size: int):
        self.num_blocks = num_blocks
        self.num_heads = num_heads
        self.head_dim = head_dim
        self.block_size = block_size
        
        # 模拟 GPU 显存中的 KV Cache (按块组织)
        # Shape: [num_blocks, num_heads, block_size, head_dim]
        self.K_cache = np.random.randn(num_blocks, num_heads, block_size, head_dim).astype(np.float32)
        self.V_cache = np.random.randn(num_blocks, num_heads, block_size, head_dim).astype(np.float32)
    
    def attention(self, Q: np.ndarray, block_table: List[int], context_len: int) -> np.ndarray:
        """
        对一个序列执行 PagedAttention。
        
        Args:
            Q:           [num_heads, head_dim]
            block_table: [num_logical_blocks] → 每个元素是物理块 ID
            context_len: 该序列已有的 token 数
        Returns:
            output: [num_heads, head_dim]
        """
        output = np.zeros((self.num_heads, self.head_dim), dtype=np.float32)
        scale = 1.0 / np.sqrt(self.head_dim)
        
        for head in range(self.num_heads):
            q = Q[head]  # [head_dim]
            
            # Online softmax 的流式累加
            max_logit = -np.inf
            sum_exp = 0.0
            attn_out = np.zeros(self.head_dim, dtype=np.float32)
            
            tokens_processed = 0
            for block_idx, physical_block in enumerate(block_table):
                # === 关键：间接寻址 ===
                # 类似 GPU kernel 的:
                # physical_block = block_table[seq_id * max_blocks + block_idx]
                K_block = self.K_cache[physical_block, head]  # [block_size, head_dim]
                V_block = self.V_cache[physical_block, head]  # [block_size, head_dim]
                
                for t in range(self.block_size):
                    if tokens_processed >= context_len:
                        break
                    
                    k = K_block[t]  # [head_dim]
                    v = V_block[t]  # [head_dim]
                    
                    qk = np.dot(q, k) * scale
                    
                    # Online softmax (数值稳定)
                    new_max = max(max_logit, qk)
                    correction = np.exp(max_logit - new_max)
                    sum_exp = sum_exp * correction + np.exp(qk - new_max)
                    max_logit = new_max
                    
                    tokens_processed += 1
                
                if tokens_processed >= context_len:
                    break
            
            # 完整的 attention 需要重新遍历做加权和（简化版）
            # 实际 GPU kernel 会用 flash attention 的方式流式计算
            # 这里省略了重新遍历的步骤
        
        return output
    
    def compare_addressing(self, block_table: List[int], context_len: int):
        """对比直接寻址 vs block_table 间接寻址的内存访问模式"""
        print(f"Context length: {context_len} tokens")
        print(f"Block size: {self.block_size}")
        print(f"Block table: {block_table}")
        print()
        
        # 直接寻址（传统方式）
        print("传统连续寻址:")
        print(f"  K[0:{context_len}, head, :]")
        print(f"  → 1 次大的内存拷贝 (连续 {context_len * self.head_dim * 2} bytes)")
        print()
        
        # PagedAttention 间接寻址
        print("PagedAttention 间接寻址:")
        total_bytes = 0
        for logical_idx, physical_block in enumerate(block_table):
            tokens_in_block = min(self.block_size, context_len - logical_idx * self.block_size)
            block_bytes = tokens_in_block * self.head_dim * 2
            total_bytes += block_bytes
            print(f"  block[{logical_idx}] → physical[{physical_block}]: "
                  f"{tokens_in_block} tokens, {block_bytes} bytes")
        print(f"  → {len(block_table)} 次小块内存拷贝, 总计 {total_bytes} bytes")
        print(f"  → 多了 {len(block_table)-1} 次 block_table 查表，但换取灵活分配")

sim = PagedAttentionSimulator(num_blocks=100, num_heads=4, head_dim=64, block_size=16)

# 模拟一个序列：逻辑块 0,1,2 映射到物理块 7, 23, 41
sim.compare_addressing(block_table=[7, 23, 41], context_len=40)


## 4. Continuous Batching：调度器的灵魂

有了 PagedAttention 的灵活内存管理，Continuous Batching 才能高效工作。

### 4.1 调度流程

```
每个推理 step:

  1. Scheduler.schedule()
     ├─ 从 WAITING 队列取出请求 (受 max_num_seqs, max_num_batched_tokens 限制)
     ├─ 为每个请求分配/检查 KV Cache blocks
     ├─ 如果 OOM → 触发抢占 (preemption)
     └─ 构建 ScheduledSequenceGroup

  2. ModelRunner.execute_model()
     ├─ prepare_input(): 将所有请求的 token 拼接成 batch
     │   ├─ input_ids:  [req0_last_token, req1_last_token, req2_last_token, ...]
     │   ├─ positions:  [req0_pos, req1_pos, req2_pos, ...]
     │   └─ block_tables: [[r0_b0, r0_b1, ...], [r1_b0, r1_b1, ...], ...]
     │
     ├─ model.forward():  一次 forward，处理整个 batch
     │   └─ PagedAttention 内部：每个序列用自己的 block_table
     │
     └─ sample(): 每个序列从自己的 logits 采样下一个 token

  3. Scheduler.update()
     └─ 将新 token 加入序列
        ├─ 如果 token == EOS → 标记 FINISHED → 释放 block_table
        └─ 如果达到 max_tokens → 标记 FINISHED
```

### 4.2 抢占 (Preemption)

当 GPU 显存的物理块用完时，需要从 RUNNING 序列中"抢占"：

```python
# 来源: vllm/core/scheduler.py

class PreemptionMode(Enum):
    RECOMPUTE = "recompute"  # 丢弃 KV Cache，需要时重新计算
    SWAP = "swap"            # 把 KV Cache 换出到 CPU 内存

class Scheduler:
    def _preempt(self, seq_group, blocks_to_free: int):
        """
        抢占一个序列，释放其物理块。
        
        策略：
        1. 优先 SWAP：将 KV blocks 拷贝到 CPU 内存 (慢但保留进度)
        2. 如果 CPU swap 空间也不够 → RECOMPUTE：丢弃，再调度时重新 prefill
        """
        if self._can_swap(seq_group, blocks_to_free):
            # SWAP 模式：GPU → CPU 拷贝
            self.block_manager.swap_out(seq_group)
            # seq_group 状态: RUNNING → SWAPPED
            # 恢复时 swap_in: CPU → GPU，继续生成
        else:
            # RECOMPUTE 模式：直接丢弃
            self.block_manager.free(seq_group)
            # seq_group 状态: RUNNING → WAITING
            # 恢复时重新 prefill，重算所有 KV Cache
```

**RECOMPUTE 的代价不是浪费，而是权衡**：
- SWAP 慢（PCIe 带宽 ~32 GB/s，搬 4GB KV Cache 需要 0.125s）但不浪费计算
- RECOMPUTE 快（GPU 上重算 prefill ~0.01s 但浪费了之前的计算结果）
- 对于短上下文（< 512 tokens）：RECOMPUTE 更划算
- 对于长上下文（> 8K tokens）：SWAP 更划算

In [ ]:
# 模拟 vLLM 调度器的一个 step
from dataclasses import dataclass
from typing import List, Deque
from collections import deque
import random

@dataclass
class Request:
    id: int
    prompt: List[int]  # token ids
    max_tokens: int
    generated: List[int] = None
    status: str = "WAITING"  # WAITING, RUNNING, FINISHED
    block_table: List[int] = None
    
    def __post_init__(self):
        self.generated = []
        self.block_table = []

class SchedulerSimulator:
    """模拟 vLLM 调度器"""
    
    def __init__(self, max_seqs: int = 8, max_batched_tokens: int = 256, block_size: int = 16):
        self.max_seqs = max_seqs
        self.max_batched_tokens = max_batched_tokens
        self.block_size = block_size
        
        # 队列
        self.waiting: Deque[Request] = deque()
        self.running: List[Request] = []
        self.finished: List[Request] = []
        
        # 全局块管理 (简化)
        self.total_blocks = 500
        self.free_blocks = deque(range(self.total_blocks))
        
        self.step_count = 0
        self.stats = {"total_tokens_generated": 0, "preemptions": 0}
    
    def add_request(self, prompt_len: int, max_tokens: int):
        """添加一个请求到等待队列"""
        req_id = len(self.waiting) + len(self.running) + len(self.finished)
        prompt = list(range(prompt_len))  # 用虚拟 token IDs 代替
        self.waiting.append(Request(id=req_id, prompt=prompt, max_tokens=max_tokens))
    
    def schedule(self) -> List[Request]:
        """
        每个 step 的调度决策。
        对应 vllm/core/scheduler.py 的 _schedule()
        """
        scheduled = []
        
        # 1. 先尝试从 WAITING 拉入新请求
        while self.waiting and len(self.running) < self.max_seqs:
            req = self.waiting[0]
            
            # 计算需要的块数
            total_tokens = len(req.prompt) + len(req.generated)
            blocks_needed = (total_tokens + self.block_size - 1) // self.block_size
            blocks_have = len(req.block_table)
            new_blocks = blocks_needed - blocks_have
            
            if len(self.free_blocks) >= new_blocks:
                # 分配新块
                for _ in range(new_blocks):
                    req.block_table.append(self.free_blocks.popleft())
                
                self.waiting.popleft()
                self.running.append(req)
                req.status = "RUNNING"
                scheduled.append(req)
            else:
                # OOM — 不走抢占逻辑，直接 break
                break
        
        # 2. 所有 RUNNING 请求也参与这一步
        for req in self.running:
            if req not in scheduled:
                scheduled.append(req)
        
        return scheduled
    
    def step(self):
        """模拟一个推理步"""
        self.step_count += 1
        
        scheduled = self.schedule()
        
        if not scheduled:
            return
        
        # 模拟: 每个 RUNNING 请求生成一个 token
        # (实际 vLLM 中是做一次 model.forward over batch)
        for req in scheduled:
            if req.status == "RUNNING":
                token = random.randint(0, 32000)  # 虚拟采样
                req.generated.append(token)
                self.stats["total_tokens_generated"] += 1
                
                # 检查是否需要新块
                total_tokens = len(req.prompt) + len(req.generated)
                blocks_needed = (total_tokens + self.block_size - 1) // self.block_size
                if blocks_needed > len(req.block_table):
                    if self.free_blocks:
                        req.block_table.append(self.free_blocks.popleft())
                    else:
                        # 需要抢占！(简化处理)
                        self.stats["preemptions"] += 1
                
                # 终止条件
                if len(req.generated) >= req.max_tokens:
                    self._finish(req)
    
    def _finish(self, req: Request):
        """请求完成，释放块"""
        req.status = "FINISHED"
        self.running.remove(req)
        self.finished.append(req)
        for block_id in req.block_table:
            self.free_blocks.append(block_id)
        req.block_table = []
    
    def run(self, steps: int = 50):
        """运行指定步数"""
        for _ in range(steps):
            self.step()
            
            if not self.running and not self.waiting:
                print(f"All requests finished at step {self.step_count}")
                break
        
        print(f"\n=== 调度统计 ===")
        print(f"Steps: {self.step_count}")
        print(f"Total tokens generated: {self.stats['total_tokens_generated']}")
        print(f"Preemptions: {self.stats['preemptions']}")
        print(f"Running: {len(self.running)}, Waiting: {len(self.waiting)}, Finished: {len(self.finished)}")
        print(f"Free blocks: {len(self.free_blocks)}/{self.total_blocks}")
        print(f"Block usage: {(1 - len(self.free_blocks)/self.total_blocks)*100:.1f}%")

# 演示
sched = SchedulerSimulator(max_seqs=8)

# 添加混合长度的请求
sched.add_request(prompt_len=128, max_tokens=20)   # 短请求
sched.add_request(prompt_len=4096, max_tokens=50)  # 长上下文
sched.add_request(prompt_len=64, max_tokens=10)
sched.add_request(prompt_len=256, max_tokens=30)

sched.run(steps=60)
print("\n💡 这就是 Continuous Batching: 短请求不会阻塞长请求，")
print("   请求完成立即释放显存给等待队列中的新请求。")


## 5. Prefix Caching：RadixAttention 的 vLLM 实现

vLLM 的 Automatic Prefix Caching (APC) 自动检测和重用公共前缀。

### 5.1 原理

```
场景：100 个用户同时在用你的 chatbot，每个人的对话都以相同的 system prompt 开头。

传统处理:
  system_prompt = "You are a helpful assistant. Answer the user's questions accurately."
  User 1: system_prompt + "What is Python?"       → prefill 需要计算
  User 2: system_prompt + "How to write a loop?"   → prefill 需要计算（又算一遍！）
  User 3: system_prompt + "Explain recursion"      → prefill 需要计算（又算一遍！）
  
  system_prompt 的 KV Cache 被重复计算了 100 次。浪费！

vLLM Prefix Caching:
  1. 用 token 序列的 hash 标识"前缀"
  2. system_prompt 的 KV blocks (比如前 4 个块) 只存一份
  3. 新请求到来时：
     a. 计算 prompt 的 hash
     b. 查找是否已有匹配的前缀块 → 找到 system_prompt 的 4 个块
     c. Fork 这些块（block_table 指向相同的物理块）
     d. 只需要 prefill 剩余的部分（用户消息）

关键数据结构 (vllm/core/block/prefix_caching_block.py):

  class PrefixCachingBlockAllocator:
      # hash → block_id 的映射
      cached_blocks: Dict[BlockHash, List[BlockId]]
      
      def get_cached_block(self, hash_val: int) -> Optional[BlockId]:
          """查找是否已有相同内容的 KV block"""
          if hash_val in self.cached_blocks and self.cached_blocks[hash_val]:
              return self.cached_blocks[hash_val][0]
          return None
```

In [ ]:
# 演示 Prefix Caching 的效果
import hashlib

def compute_block_hash(token_ids: List[int]) -> int:
    """
    计算一个 block 的 content hash。
    实际 vLLM 用更高效的 rolling hash，这里用 SHA256 示意。
    """
    data = bytes(str(token_ids), 'utf-8')
    return int.from_bytes(hashlib.sha256(data).digest()[:8], 'big')

# 模拟 system prompt 的 token IDs
system_prompt = [101, 2017, 2021, 2003, 2107, 4055, 1012, 1005]  # 示意

class PrefixCache:
    """简化版 Prefix Caching"""
    def __init__(self):
        self.hash_to_block: Dict[int, int] = {}  # hash → physical_block_id
        self.hit_count = 0
        self.miss_count = 0
        self.compute_saved = 0  # 节省的 token 计算量
    
    def lookup(self, tokens: List[int], block_size: int = 16) -> int:
        """
        查找多少个前缀块已被缓存。
        返回命中的 token 数。
        """
        matched_tokens = 0
        for i in range(0, len(tokens), block_size):
            block = tokens[i:i+block_size]
            block_hash = compute_block_hash(block)
            
            if block_hash in self.hash_to_block:
                matched_tokens += len(block)
                self.hit_count += 1
            else:
                # Cache 当前块，让后续请求能命中
                block_id = len(self.hash_to_block)
                self.hash_to_block[block_hash] = block_id
                self.miss_count += 1
                break  # 第一个不匹配的块之后不继续
        
        self.compute_saved += matched_tokens
        return matched_tokens

cache = PrefixCache()

# 模拟 20 个请求，都有相同的 64-token system prompt，各自有不同的 16-token user message
for i in range(20):
    user_msg = list(range(200 + i*16, 200 + i*16 + 16))
    full_prompt = system_prompt + user_msg
    
    matched = cache.lookup(full_prompt)
    if i < 3:
        print(f"  Request {i}: {matched}/{len(full_prompt)} tokens hit cache ({matched/len(full_prompt)*100:.0f}%)")

print(f"\n=== Prefix Cache 统计 ===")
print(f"Hit: {cache.hit_count}, Miss: {cache.miss_count}")
print(f"Hit rate: {cache.hit_count/(cache.hit_count+cache.miss_count)*100:.1f}%")
print(f"Computation saved: {cache.compute_saved} token-prefills avoided")
print(f"\n💡 system_prompt 的 4 个 block 只被 prefill 了一次")
print(f"   后续 19 个请求直接复用 → 节省 19×64=1216 个 token 的 prefill 计算")


## 6. vLLM vs llama.cpp：两种哲学的终极对比

两个项目都极其成功，但设计哲学截然不同：

| 维度 | llama.cpp | vLLM |
|------|-----------|------|
| **目标场景** | 个人电脑，1-2 并发 | 服务器，100+ 并发 |
| **核心创新** | GGUF 量化 + ggml 后端 | PagedAttention + Continuous Batching |
| **语言** | C/C++ (性能) | Python + CUDA (生态) |
| **后端** | CPU/CUDA/Metal/Vulkan 手写 | 依赖 PyTorch + CUDA |
| **内存管理** | 预分配 + mmap | 分页 + 块表 + COW |
| **KV Cache 共享** | ❌ 不支持 | ✅ 支持 (fork 语义) |
| **量化** | 自研 K-quant (最佳 CPU 量化) | 委托给 AWQ/GPTQ/FP8 |
| **安装复杂度** | `make && ./llama-cli` | `pip install vllm` (需要 CUDA 12.x) |
| **适用模型** | GGUF 格式模型 | HuggingFace 格式模型 |

### 一句话总结

- **学底层原理** → llama.cpp：GGUF 格式、K-quant 量化、SIMD 内核、Metal 后端——全部开源，C 代码直接读
- **搭建生产服务** → vLLM：PagedAttention 解决显存碎片，Continuous Batching 榨干 GPU 利用率，
  Prefix Caching 节省重复计算，OpenAI 兼容 API 开箱即用

两个项目都是"做一件事并做到极致"的典范。理解它们就理解了 LLM 推理的完整技术栈。

## 下一步

- → `04-tensorrt-llm-deep-dive.ipynb`：编译优化的极限——如何用 TensorRT 将推理加速 5×？
- → `06-sglang-deep-dive.ipynb`：RadixAttention——比 Prefix Caching 更进一步的前缀复用
- → 返回 `../00-overview.ipynb` 查看完整的推理服务全景

In [ ]:
# 最终实验：对比两种 KV Cache 策略的显存利用率

def compare_memory_efficiency():
    """
    模拟场景：server 端同时服务 10 个请求
    - 每个请求的 system prompt = 128 tokens (共享)
    - 每个请求平均生成 256 tokens
    """
    n_requests = 10
    system_prompt_len = 128
    avg_generated = 256
    block_size = 16
    max_len = 4096
    
    # 传统预分配
    kv_per_token_kb = 128  # 假设: KV Cache 每个 token 占用 128KB (Llama-7B 典型值)
    traditional_mem = n_requests * max_len * kv_per_token_kb / 1024  # MB
    
    # PagedAttention 按需分配
    actual_tokens_per_req = system_prompt_len + avg_generated
    blocks_per_req = (actual_tokens_per_req + block_size - 1) // block_size
    # system prompt 的 8 个 block 共享，其余 10 个请求各自分配
    system_blocks = (system_prompt_len + block_size - 1) // block_size  # 8 块
    unique_blocks_per_req = blocks_per_req - system_blocks  # 每人另需 (384-128)/16=16 块
    
    shared_mem = system_blocks * block_size * kv_per_token_kb / 1024
    unique_mem = n_requests * unique_blocks_per_req * block_size * kv_per_token_kb / 1024
    paged_mem = shared_mem + unique_mem
    
    print(f"=== KV Cache 显存占用对比 ({n_requests} 个并发请求) ===\n")
    print(f"传统预分配 (max_len={max_len}):")
    print(f"  每个请求: {max_len * kv_per_token_kb / 1024:.0f} MB")
    print(f"  总计: {traditional_mem:.0f} MB")
    print()
    print(f"PagedAttention (按需分配 + 前缀共享):")
    print(f"  共享前缀 (system prompt): {shared_mem:.0f} MB (只存一份)")
    print(f"  各自生成部分: {unique_mem:.0f} MB")
    print(f"  总计: {paged_mem:.0f} MB")
    print()
    print(f"💡 节省: {(1 - paged_mem/traditional_mem)*100:.0f}% ({(traditional_mem/paged_mem):.1f}× 效率提升)")
    print(f"   等效可服务请求数: {int(n_requests * traditional_mem / paged_mem)} 个")

compare_memory_efficiency()
